<br><br>
<h1><font color="#B30033" size=5>Sistemas Inteligentes - Curso 2026-2027</font></h1>

<h1><font color="#B30033" size=5>Práctica 2: Aprendizaje por refuerzo</font></h1>

<br>
<div style="text-align: left">
<font color="#4E70BE" size=3>Profesorado:</font><br>
<ul>
  <li><font color="#4E70BE" size=3>Mª Julia Flores Gallego (Julia.Flores@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>José Antonio Gámez Martín (Jose.Gamez@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>Ismael García Varea (Ismael.Garcia@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>José Roldán Gómez (Jose.Roldan@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>Bastián Andrés Troncoso Retamales (Bastian.Troncoso@uclm.es)</font><br></li>
  <li><font color="#4E70BE" size=3>Teresa Valero Díaz (Teresa.Valero@uclm.es)</font><br></li>
</ul>
</div>

<div style="text-align: left">
<font color="#4E70BE" size=3>Autores de la entrega:</font><br>
<ul>
  <li><font color="#4E70BE" size=3>Nombre y apellidos, correo</font><br></li>
  <li><font color="#4E70BE" size=3>Nombre y apellidos, correo</font><br></li>
</ul>
</div>

--------------------


## 1. Introducción

En la PR1 el agente lo sabía todo: el mapa, los costes y a dónde lleva cada acción. Con esa
información, encontrar el mejor camino es planificar, y A$^*$ lo resuelve.

Esta práctica quita esa información. **El agente no ve el plano de la estación**: solo puede
actuar y observar dónde acaba y qué recompensa recibe. Además, el movimiento ya no es fiable:
la acción elegida se ejecuta la mayoría de las veces, pero no siempre.

Sin modelo del mundo no se puede planificar, así que hay que **aprender de la experiencia**:
probar, equivocarse y quedarse con lo que funciona. Eso es el aprendizaje por refuerzo, y lo
que se aprende no es un camino sino una **política**: qué hacer en cada posición.

¿Y cómo saber si lo aprendido es bueno? En la segunda parte calcularás la **política óptima**
de verdad con **iteración de valores**, que sí usa el modelo del mundo (las probabilidades de
cada transición), y medirás cuánto se le acerca lo que aprende Q-learning sin él.

Al terminar deberías saber:

* **Modelar** una tarea como un problema de decisión de Markov: estados, acciones, dinámica
  y recompensas.
* **Implementar** Q-learning tabular con una política de exploración $\varepsilon$-voraz.
* **Calcular la política óptima** con iteración de valores cuando se conoce el modelo, y
  comparar con ella lo que se aprende sin modelo.
* **Ajustar** $\alpha$, $\gamma$ y el esquema de decaimiento de $\varepsilon$, y explicar qué
  hace cada uno.
* **Diseñar recompensas** y reconocer cuándo una recompensa mal puesta produce un agente que
  hace justo lo que no queremos.
* **Evaluar una política**, que no es lo mismo que mirar la tabla Q.


## 2. Descripción del problema

La estación es la misma de la PR1, con sus plantas, sus mamparos, sus zonas caras y sus
escotillas. Lo que cambia es lo que el agente sabe y lo que ocurre cuando actúa.

| Elemento | En esta práctica |
|---|---|
| Estado | La posición `[nivel, fila, columna]` donde está el agente |
| Acciones | `NORTE`, `ESTE`, `SUR`, `OESTE`, `SUBIR`, `BAJAR` |
| Dinámica | Desconocida para el agente, y **estocástica** |
| Recompensa | La que devuelve el entorno en cada paso |
| Episodio | Empieza en el origen y acaba al llegar al destino o al agotar el límite de pasos |

### 2.1. El movimiento falla

Con probabilidad `p_exito` se ejecuta la acción elegida. En caso contrario:

* si la acción era **horizontal**, el agente **resbala** y se ejecuta otra horizontal al azar;
* si era `SUBIR` o `BAJAR`, **la escotilla se atasca** y el agente se queda donde está.

Y si la acción que finalmente se ejecuta no es posible —hay un mamparo, o no hay escotilla—,
el agente **choca**: no se mueve y cobra la recompensa de colisión.

Además, cada episodio tiene un **límite de pasos** (`limite_pasos`): si el agente no ha llegado
al destino cuando lo agota, el episodio se corta. Es 4 × filas × columnas de la estación: 400
pasos en `PR2-S-01` y 1 600 en las instancias M. Como `p_exito`, forma parte de la dinámica y
viene en la instancia, en `consulta.dinamica`. No es una recompensa, y cortar un episodio por
tiempo no es llegar a un estado final: es solo dejar de mirar.

### 2.2. Las recompensas

Vienen en la instancia, en `consulta.dinamica.recompensas`:

| Recompensa | Valor por defecto | Cuándo |
|---|---|---|
| `objetivo` | +100 | Al llegar al destino |
| `paso_por_coste` | −1 | Por cada unidad de coste del movimiento ejecutado |
| `colision` | −5 | Al chocar contra un mamparo o una escotilla que no existe |

Un paso por un pasillo de coste 1 da −1; por un conducto técnico de coste 3, −3. Así el agente
no solo aprende a llegar: aprende a llegar barato, que es justo lo que optimizaba la PR1.

**Estas recompensas se pueden cambiar**, y hacerlo es parte de la práctica (sección 9).


## 3. Desarrollo de la práctica

Trabajarás sobre tres instancias del catálogo, en `catalogo/pr2/`:

| Instancia | Estados | Coste óptimo determinista (PR1) |
|---|---|---|
| `PR2-S-01` | 87 | 21 |
| `PR2-M-01` | 829 | 44 |
| `PR2-M-02` | 822 | 48 |

Esa última columna es el coste del mejor camino **si el movimiento no fallara**, calculado con
el algoritmo de la PR1. Conviene adelantar el final: tu política no va a alcanzarlo, y no
porque aprenda mal. La referencia buena es la de la **política óptima del problema con
resbalones**, y la calcularás tú con iteración de valores (sección 7.4).

### 3.1. El entorno

Se te proporciona en `ssii_estacion.entorno`. Tiene dos métodos, y con eso basta:

```python
estado = entorno.reset()                       # empieza un episodio
estado, recompensa, terminado, info = entorno.paso(accion)
```

El entorno **no expone la estación**. El atributo donde la guarda empieza por guion bajo, y
leerlo es hacer trampa: si pudieras consultar el mapa, la política óptima se calcularía con
A$^*$ y no habría práctica. En la corrección se comprueba.

Para la iteración de valores el paquete te da, aparte, el **modelo** del problema,
`ModeloEstacion`: para cada estado y acción, las transiciones posibles con su probabilidad y
su recompensa. Es exactamente lo que el agente de Q-learning no tiene, así que **solo se usa
en iteración de valores**; usarlo en Q-learning también se detecta en la corrección.

### 3.2. Qué implementas tú

1. `q_learning(entorno, episodios, ...)` — Q-learning tabular con $\varepsilon$-voraz.
   Devuelve la tabla `Q` y la **curva de aprendizaje**.
2. `politica_de(Q)` — la política voraz respecto a Q: en cada estado, la acción de mayor valor.
3. `iteracion_valores(modelo, gamma, umbral)` — iteración de valores sobre el modelo.
   Devuelve las utilidades óptimas `U` y el historial de convergencia.
4. `politica_desde_valores(modelo, U, gamma)` — la política voraz respecto a U: la óptima.

El resto —evaluación de la política, formato de la solución, validación— lo pone el paquete,
para que los números de todas las entregas sean comparables.


## 4. Plan de trabajo

Hay cuatro sesiones de laboratorio. Una forma razonable de repartirlas:

1. **Sesión 1**: el entorno y Q-learning. Ejecuta episodios con acciones al azar, mira qué
   devuelve `paso` y comprueba cuántas veces la acción ejecutada no es la elegida.
   Implementa Q-learning, consigue que aprenda en `PR2-S-01` y mira su política (7.1-7.3).
2. **Sesión 2**: iteración de valores. Impleméntala con el modelo, obtén la política óptima
   de `PR2-S-01` y compárala con la de Q-learning con `evaluar_politica` y con los mapas
   (7.4-7.5).
3. **Sesión 3**: hiperparámetros y escala. $\alpha$, $\gamma$, esquema de decaimiento
   de $\varepsilon$ y número de episodios en las instancias M; curvas de aprendizaje, y qué
   parte de la recompensa óptima consigue Q-learning en cada caso (9.1-9.2).
4. **Sesión 4**: diseño de recompensas, preguntas de análisis y memoria (9.3-9.4).

Merece la pena empezar siempre por `PR2-S-01`: 87 estados y unos segundos por experimento.
Depurar Q-learning en una instancia M es perder la tarde.


## 5. Entrega y evaluación

El trabajo se realiza **en parejas** (también se admite individualmente; no se autorizan
grupos de tres o más). Se entrega **este mismo cuaderno** completado, junto con los ficheros
JSON de solución que genere la sección 8.

Recuerda que las prácticas se evalúan combinando dos factores:

1. **25 %** — entrega del código y rendimiento sobre una batería de pruebas automáticas.
2. **75 %** — examen de prácticas en laboratorio, en la última semana de clase.

Se requiere un **mínimo de 4 sobre 10** en la media ponderada para superar o compensar esta
parte de la asignatura.

Si sigues la **evaluación no continua**, entregas además la ampliación de la sección 10:
dispones de más tiempo para resolver y entregar la práctica. Quien sigue la evaluación
continua no tiene que entregarla.

### 5.1. Corrección automática

Aquí no hay un único resultado correcto: dos ejecuciones con semillas distintas aprenden
políticas distintas, y las dos pueden ser buenas. Por eso lo que se comprueba es el
**comportamiento de la política entregada**, no la tabla Q ni el número de episodios:

* que la política use acciones válidas y cubra los estados por los que pasa;
* que, al ejecutarla con `evaluar_politica` y la semilla de corrección, alcance el destino
  de forma consistente;
* que los números que declaras en el fichero de solución coincidan con los que salen al
  reevaluar tu política;
* que iteración de valores calcule los valores óptimos: se comparan con los de referencia;
* que Q-learning no haya mirado el mapa ni usado el modelo.

Parte de las comprobaciones las tienes a mano. Desde la carpeta de la práctica:

```bash
pip install pytest
python3 -m pytest PR2/pruebas -q
```

Pasarlas es el mínimo, no la nota: en la corrección se añaden otras que no ves.

### 5.2. Sobre el uso de herramientas de IA generativa

Q-learning tabular es de los algoritmos que un asistente escribe de memoria, así que conviene
decirlo claro: **la entrega supone la declaración de autoría propia**, y en el examen de
prácticas —que vale el 75 % de esta parte— tendrás que trabajar sin ayuda sobre este mismo
código. Se aplica la política de la asignatura descrita en el tema 0.


## 6. El entorno en funcionamiento

Empezamos comprobando que todo carga y viendo qué se siente al actuar sin plano.

In [ ]:
# El paquete ssii_estacion está en la carpeta superior, junto al catálogo.
import sys, os
sys.path.append(os.path.abspath(".."))

import random
import matplotlib.pyplot as plt

from ssii_estacion import solucion_pr2, guardar_solucion
from ssii_estacion.entorno import EntornoEstacion, ModeloEstacion, evaluar_politica
from ssii_estacion.vis import dibujar_politica

print("paquete cargado correctamente")

In [ ]:
# Un episodio con acciones al azar: así se ve el mundo desde dentro
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json", semilla=1)
print(entorno)

rnd = random.Random(0)
estado = entorno.reset()
print("estado inicial:", estado)

for k in range(8):
    accion = rnd.choice(entorno.acciones)
    estado, recompensa, terminado, info = entorno.paso(accion)
    print(f"  elegida {accion:6} -> ejecutada {str(info['accion_ejecutada']):6} "
          f"| estado {estado} | r {recompensa:6.1f} | choque {info['choque']}")
    if terminado:
        break

In [ ]:
# ¿Cuántas veces se ejecuta de verdad la acción elegida?
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json", semilla=7)
entorno.reset()
iguales = 0
N = 2000
for _ in range(N):
    accion = "ESTE"
    _, _, terminado, info = entorno.paso(accion)
    iguales += (info["accion_ejecutada"] == accion)
    if terminado:
        entorno.reset()
print(f"la acción elegida se ejecutó el {iguales / N:.1%} de las veces "
      f"(p_exito = {entorno.p_exito})")

### 6.1. Resumen de la API

```python
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json", semilla=1)

entorno.acciones          # ('NORTE', 'ESTE', 'SUR', 'OESTE', 'SUBIR', 'BAJAR')
entorno.p_exito           # probabilidad de que la acción elegida sea la ejecutada
entorno.limite_pasos      # pasos máximos por episodio (parte de la dinámica)
entorno.recompensas       # dict con objetivo, paso_por_coste y colision

estado = entorno.reset()                                  # -> Pos(nivel, fila, col)
estado, recompensa, terminado, info = entorno.paso(accion)
# info: exito, coste, choque, accion_ejecutada, pasos

evaluar_politica(entorno, politica, episodios=200, semilla=12345)
# -> tasa_exito, recompensa_media, recompensa_desv, pasos_medios, coste_medio
```

Para experimentar con el diseño de recompensas (sección 9), el constructor las acepta (y
también `p_exito` y `limite_pasos`, aunque en la práctica no hace falta cambiarlos):

```python
entorno = EntornoEstacion.desde_instancia(ruta, semilla=1,
                                          recompensas={"paso_por_coste": -2})
```

Los estados son `Pos`, es decir, tuplas con nombre: sirven como clave de un diccionario, que
es todo lo que necesita una tabla Q.

Y el modelo, **solo para iteración de valores** (sección 7.4):

```python
modelo = ModeloEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json")

modelo.estados            # todas las posiciones transitables
modelo.acciones           # las seis acciones
modelo.origen, modelo.destino
modelo.es_terminal(s)     # True solo en el destino
modelo.transiciones(s, a) # [(probabilidad, s2, recompensa), ...]; las probabilidades suman 1
```

## 7. Implementación

### 7.1. Q-learning

La regla de actualización es la del tema 5:

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,\big]$$

Y la exploración es $\varepsilon$-voraz: con probabilidad $\varepsilon$ una acción al azar, y
en caso contrario la mejor según Q. $\varepsilon$ **decrece** a lo largo del entrenamiento:
al principio conviene explorar, al final explotar lo aprendido.

Fíjate en que `Q` es un diccionario de diccionarios y **no se inicializa con todos los
estados**: no los conoces. Se van creando según aparecen.

In [ ]:
def q_learning(entorno, episodios, alpha=0.2, gamma=0.95,
               eps0=1.0, eps_min=0.05, fraccion_decaimiento=0.8,
               semilla=0, ventana=100):
    """Q-learning tabular con epsilon-voraz decreciente.

    Devuelve (Q, curva), donde curva[i] es la recompensa media de la i-ésima
    ventana de episodios: es lo que se dibuja para ver si el agente aprende.
    """
    rnd = random.Random(semilla)
    acciones = entorno.acciones
    Q = {}
    recompensas, curva = [], []

    for ep in range(episodios):
        # TODO: calcular epsilon para este episodio (decreciente hasta eps_min)
        eps = None

        s = entorno.reset()
        total = 0.0
        while True:
            # TODO: asegurar que Q tiene una entrada para s (todas las acciones a 0.0)
            # TODO: elegir accion con epsilon-voraz
            accion = None

            # TODO: ejecutar la accion en el entorno

            # TODO: actualizar Q[s][accion] con la regla de Q-learning
            #       ojo: el estado siguiente también tiene que existir en Q

            # TODO: acumular la recompensa y avanzar al estado siguiente
            break   # <- quita esto cuando implementes el bucle

        recompensas.append(total)
        if (ep + 1) % ventana == 0:
            curva.append(round(sum(recompensas[-ventana:]) / ventana, 2))

    return Q, curva


def politica_de(Q):
    """La política voraz respecto a Q: en cada estado, la acción de mayor valor."""
    # TODO: devolver un diccionario estado -> acción
    raise NotImplementedError

### 7.2. Comprobación rápida

Sobre la instancia pequeña, unos miles de episodios bastan. Además de la tabla Q,
`q_learning` devuelve la **curva de aprendizaje**: la recompensa media por episodio en
bloques de 100 episodios consecutivos (es el parámetro `ventana=100`). Así, `curva[0]` es la
media de los episodios 1 a 100, `curva[1]` la de los episodios 101 a 200, y así
sucesivamente.

Si los últimos valores de la curva no quedan claramente por encima de cero, el agente no
está aprendiendo a llegar al destino: revísalo aquí antes de pasar a las instancias M.

In [ ]:
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json", semilla=1)
Q, curva = q_learning(entorno, 3000, semilla=1)

print(f"estados visitados: {len(Q)}")
print(f"curva: {curva[0]} -> {curva[len(curva)//2]} -> {curva[-1]}")

politica = politica_de(Q)
print(evaluar_politica(entorno, politica))

### 7.3. Ver la política

Una política no es un camino: es una acción para **cada** posición, porque el robot puede
acabar en cualquier sitio cuando resbala. `dibujar_politica` la pinta sobre el plano: una
flecha en cada posición con la acción que elige (▲ SUBIR, ▼ BAJAR) y, en gris, las
posiciones que el agente no ha pisado nunca. El plano lo pone el paquete solo para el
dibujo: tu agente sigue sin verlo.

Sigue las flechas desde el origen: deberías llegar al destino. Mira también las posiciones
alejadas de ese recorrido: ¿tienen sentido sus flechas?

In [ ]:
dibujar_politica(entorno, politica, titulo="PR2-S-01 — política aprendida");

### 7.4. Iteración de valores: la política óptima con el modelo

Q-learning aprende sin saber qué hace cada acción. Si se conoce el modelo, la política óptima
se puede **calcular**. Iteración de valores aplica una y otra vez la actualización de Bellman
del tema 5 a todos los estados:

$$U_{i+1}(s) \leftarrow \max_a \Big( R(s,a) + \gamma \sum_{s'} T(s,a,s')\, U_i(s') \Big)$$

hasta que el mayor cambio de una pasada, $\delta$, baja de un umbral. La política óptima es la
voraz respecto a $U$:

$$\pi^{*}(s) = \arg\max_a \Big( R(s,a) + \gamma \sum_{s'} T(s,a,s')\, U(s') \Big)$$

**$T$ y $R$ salen del modelo.** `modelo.transiciones(s, a)` devuelve la lista de resultados
posibles de ejecutar $a$ en $s$, cada uno como una terna `(p, s2, r)`: la probabilidad, el
estado al que se llega y la recompensa que se recibe. Por tanto:

* $T(s,a,s')$ es la probabilidad `p` de la terna que llega a $s'$;
* $R(s,a)$ es la recompensa **esperada** al ejecutar $a$ en $s$: $\sum p \cdot r$ sobre las
  ternas. Depende de la acción, como dice el tema: no cuesta lo mismo moverse por un pasillo
  que chocar o que subir por una escotilla.

Tres detalles:

* El destino es el único estado terminal. Como el +100 de llegar ya va dentro de $R(s,a)$ de
  la acción que llega, la utilidad del destino es 0 (en el algoritmo del tema,
  $U(s) \leftarrow R(s)$ para los estados terminales, y aquí $R(\text{destino}) = 0$).
* El problema es de **horizonte infinito con descuento**: el límite de pasos del entorno no
  forma parte del modelo.
* Si dos acciones empatan, cualquiera vale.

In [ ]:
def iteracion_valores(modelo, gamma=0.95, umbral=1e-4, max_iteraciones=10_000):
    """Iteración de valores sobre el modelo completo de la estación.

    Devuelve (U, historial): U[s] es la utilidad óptima de cada estado (0 en el destino,
    que es terminal) e historial[k] el mayor cambio de U en la pasada k (el delta del
    tema). Termina cuando ese cambio baja de `umbral` (o al llegar a `max_iteraciones`).
    """
    U = {s: 0.0 for s in modelo.estados}
    historial = []

    # TODO: repetir pasadas hasta que el mayor cambio (delta) sea menor que umbral
    #   - en cada estado no terminal:
    #       U'(s) = max_a ( R(s,a) + gamma * sum_s' T(s,a,s') * U(s') )
    #     con las ternas (p, s2, r) de modelo.transiciones(s, a):
    #       R(s,a) = suma de p * r        y        sum_s' T(s,a,s') * U(s') = suma de p * U[s2]
    #   - guardar en historial el mayor cambio de la pasada
    raise NotImplementedError


def politica_desde_valores(modelo, U, gamma=0.95):
    """La política voraz respecto a U: en cada estado no terminal, la acción que maximiza
    R(s,a) + gamma * sum_s' T(s,a,s') * U(s')."""
    # TODO: devolver un diccionario estado -> acción (sin el destino)
    raise NotImplementedError

### 7.5. La óptima frente a la aprendida

Para comparar dos políticas, **evalúa las dos con `evaluar_politica`** sobre el mismo entorno.
No compares $U(\text{origen})$ con la recompensa media de `evaluar_politica`: $U$ es una
utilidad **descontada** y esperada, y `evaluar_politica` mide la recompensa total, sin
descontar, de 200 episodios concretos.

Los mapas ayudan a ver dónde difieren: el primero pinta la política óptima sobre $U$; el
segundo, la de Q-learning con las acciones que no coinciden con la óptima en rojo.

In [ ]:
modelo = ModeloEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json")
U, historial = iteracion_valores(modelo, gamma=0.95)
optima = politica_desde_valores(modelo, U, gamma=0.95)
print(f"{len(historial)} pasadas; último delta {historial[-1]:.1e}; U(origen) = {U[modelo.origen]:.2f}")

print("óptima    :", evaluar_politica(entorno, optima))
print("Q-learning:", evaluar_politica(entorno, politica))

dibujar_politica(modelo, optima, U=U, titulo="PR2-S-01 — política óptima (iteración de valores)")
dibujar_politica(entorno, politica, comparar_con=optima, titulo="PR2-S-01 — Q-learning frente a la óptima");

## 8. Evaluar la política (no la tabla Q)

Una tabla Q con buenos números no significa nada por sí sola: lo que importa es **cómo se
comporta la política que sale de ella**. `evaluar_politica` ejecuta 200 episodios de prueba
con una semilla fija —la misma para todas las entregas— y devuelve la tasa de éxito, la
recompensa media con su desviación, los pasos medios y el coste medio del recorrido.

Ese coste medio es comparable con el coste óptimo determinista de la PR1 y con el de la
política óptima que calculaste en el 7.4, y ahí está la conversación interesante de esta
práctica. Compáralos antes de seguir.

### 8.1. Generar los ficheros de solución

Esta celda produce los JSON que se entregan y que corrige el sistema automático. No la
modifiques: cámbiale solo la lista de autores y, si quieres, los hiperparámetros.

In [ ]:
AUTORES = ["Nombre Apellidos <correo@alu.uclm.es>",
           "Nombre Apellidos <correo@alu.uclm.es>"]

INSTANCIAS = ["PR2-S-01", "PR2-M-01", "PR2-M-02"]
EPISODIOS = {"PR2-S-01": 3000, "PR2-M-01": 8000, "PR2-M-02": 8000}

HIPER = {"alpha": 0.2, "gamma": 0.95, "eps0": 1.0, "eps_min": 0.05,
         "fraccion_decaimiento": 0.8, "semilla": 1}

import time, os

def entrenar_y_exportar(nombre_instancia, carpeta="soluciones"):
    entorno = EntornoEstacion.desde_instancia(f"../catalogo/pr2/{nombre_instancia}.json",
                                              semilla=HIPER["semilla"])
    t0 = time.perf_counter()
    Q, curva = q_learning(entorno, EPISODIOS[nombre_instancia],
                          alpha=HIPER["alpha"], gamma=HIPER["gamma"],
                          eps0=HIPER["eps0"], eps_min=HIPER["eps_min"],
                          fraccion_decaimiento=HIPER["fraccion_decaimiento"],
                          semilla=HIPER["semilla"])
    entrenamiento_s = time.perf_counter() - t0

    politica = politica_de(Q)
    ev = evaluar_politica(entorno, politica)

    doc = solucion_pr2(
        instancia_id=nombre_instancia, algoritmo="q_learning", autores=AUTORES,
        politica=politica,
        hiperparametros={**HIPER, "episodios": EPISODIOS[nombre_instancia]},
        entrenamiento={"episodios": EPISODIOS[nombre_instancia], "ventana": 100,
                       "curva": curva, "tiempo_s": round(entrenamiento_s, 2),
                       "estados_visitados": len(Q)},
        evaluacion=ev,
    )
    guardar_solucion(doc, os.path.join(carpeta, f"{nombre_instancia}.q_learning.json"))
    return {"instancia": nombre_instancia, "estados": len(Q),
            "tiempo_s": round(entrenamiento_s, 1), **ev}, curva

resultados, curvas = [], {}
for nombre in INSTANCIAS:
    print("entrenando", nombre, "...")
    fila, curva = entrenar_y_exportar(nombre)
    resultados.append(fila)
    curvas[nombre] = curva
print("\nlisto: soluciones escritas en ./soluciones/")

## 9. Estudio experimental

### 9.1. Resultados

In [ ]:
import pandas as pd

df = pd.DataFrame(resultados)
display(df[["instancia", "estados", "tiempo_s", "tasa_exito",
            "recompensa_media", "recompensa_desv", "pasos_medios", "coste_medio"]])

# Y la comparación que da sentido a la práctica: lo aprendido frente a lo óptimo
OPTIMO_PR1 = {"PR2-S-01": 21, "PR2-M-01": 44, "PR2-M-02": 48}
filas = []
for fila in resultados:
    nombre = fila["instancia"]
    ruta = f"../catalogo/pr2/{nombre}.json"
    modelo = ModeloEstacion.desde_instancia(ruta)
    U, _ = iteracion_valores(modelo, gamma=HIPER["gamma"])
    optima = evaluar_politica(EntornoEstacion.desde_instancia(ruta, semilla=HIPER["semilla"]),
                              politica_desde_valores(modelo, U, gamma=HIPER["gamma"]))
    filas.append({"instancia": nombre,
                  "recompensa Q-learning": fila["recompensa_media"],
                  "recompensa óptima": optima["recompensa_media"],
                  "coste Q-learning": fila["coste_medio"],
                  "coste óptimo (VI)": optima["coste_medio"],
                  "coste óptimo sin resbalones (PR1)": OPTIMO_PR1[nombre]})
display(pd.DataFrame(filas))

### 9.2. Curvas de aprendizaje y mapas de la política

Representa al menos:

* la curva de aprendizaje de las tres instancias;
* el efecto de $\alpha$ y de $\gamma$ en una de ellas (varias curvas en la misma gráfica);
* el efecto del esquema de decaimiento de $\varepsilon$: constante frente a decreciente;
* y, con `dibujar_politica`, los mapas de la política que se proponen después de la gráfica.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for nombre, curva in curvas.items():
    ax.plot([(i + 1) * 100 for i in range(len(curva))], curva, marker="", label=nombre)
ax.axhline(0, color="#adb5bd", lw=.8)
ax.set_xlabel("episodios")
ax.set_ylabel("recompensa media (ventanas de 100)")
ax.set_title("Curvas de aprendizaje")
ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()

La curva dice cuánto gana el agente por episodio, pero no **dónde** ha aprendido. Si a
`dibujar_politica` le das también la tabla Q, pinta de fondo el valor aprendido de cada
posición, $\max_a Q(s, a)$: las zonas claras son las que el agente ya sabe que llevan al
destino; en las oscuras, las flechas son poco más que ruido.

La primera celda compara pocos y muchos episodios en una instancia M. La segunda compara
dos entrenamientos con semillas distintas: con `comparar_con`, las acciones que difieren
salen en rojo. Las dos son el punto de partida de las preguntas 5, 7 y 8 del apartado 9.4.

In [ ]:
# El valor aprendido, con pocos y con muchos episodios
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-M-01.json", semilla=1)
for episodios in (1500, 8000):
    Q, _ = q_learning(entorno, episodios, semilla=1)
    dibujar_politica(entorno, politica_de(Q), Q=Q,
                     titulo=f"PR2-M-01 — valor y política tras {episodios} episodios");

In [ ]:
# Dos entrenamientos, dos políticas: ¿en cuántas posiciones difieren?
entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json", semilla=1)
Q1, _ = q_learning(entorno, 3000, semilla=1)
Q2, _ = q_learning(entorno, 3000, semilla=2)
p1, p2 = politica_de(Q1), politica_de(Q2)

comunes = [s for s in p1 if s in p2]
print(f"acciones distintas: {sum(p1[s] != p2[s] for s in comunes)} de {len(comunes)}")
print("semilla 1:", evaluar_politica(entorno, p1))
print("semilla 2:", evaluar_politica(entorno, p2))
dibujar_politica(entorno, p1, comparar_con=p2, titulo="PR2-S-01 — semilla 1 frente a semilla 2");

### 9.3. Diseño de recompensas

El entorno acepta otras recompensas. Prueba al menos estas tres variantes en `PR2-S-01` y
mide la política resultante con `evaluar_politica`:

1. **Sin penalizar el paso** (`paso_por_coste: 0`): ¿sigue aprendiendo a llegar? ¿Llega igual
   de barato?
2. **Premiando el paso** (`paso_por_coste: +1` y `+3`): mira la tasa de éxito y los pasos
   medios antes de sacar conclusiones. Esto es *reward hacking* de manual, y es el apartado
   que más dice en la memoria.
3. **Sin recompensa de colisión** (`colision: 0`): ¿cambia el comportamiento cerca de los
   mamparos?

Una tabla con las cinco configuraciones y un párrafo explicando qué aprendió el agente en
cada caso. Ojo con la segunda: el resultado depende también de $\gamma$, y la segunda celda
lo enseña. Piensa cuánto vale dar vueltas para siempre cobrando $r$ por paso, frente a los
+100 de llegar.

In [ ]:
VARIANTES = {
    "original":           {},
    "paso gratis":        {"paso_por_coste": 0},
    "paso premiado +1":   {"paso_por_coste": +1},
    "paso premiado +3":   {"paso_por_coste": +3},
    "choque gratis":      {"colision": 0},
}

filas = []
for nombre, cambio in VARIANTES.items():
    entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json",
                                              semilla=1, recompensas=cambio)
    Q, _ = q_learning(entorno, 3000, semilla=1)
    ev = evaluar_politica(entorno, politica_de(Q))
    filas.append({"recompensas": nombre, **ev})

display(pd.DataFrame(filas)[["recompensas", "tasa_exito", "recompensa_media",
                             "pasos_medios", "coste_medio"]])

In [ ]:
# La misma recompensa (+1 por paso) con dos gammas distintas
for gamma in (0.95, 0.99):
    entorno = EntornoEstacion.desde_instancia("../catalogo/pr2/PR2-S-01.json",
                                              semilla=1, recompensas={"paso_por_coste": +1})
    Q, _ = q_learning(entorno, 3000, gamma=gamma, semilla=1)
    ev = evaluar_politica(entorno, politica_de(Q))
    print(f"gamma={gamma}: tasa de éxito {ev['tasa_exito']:.0%}, "
          f"pasos medios {ev['pasos_medios']:.1f}, recompensa media {ev['recompensa_media']:.1f}")

### 9.4. Preguntas de análisis

Estas son las preguntas que la memoria debe responder, con tus datos delante.

1. **Tu política no alcanza el coste óptimo de la PR1.** Da los dos números para las tres
   instancias y explica por qué la diferencia no es un fallo del algoritmo. Compáralos después
   con el coste medio de la política óptima que calcula iteración de valores: ¿qué parte de
   la diferencia se debe a los resbalones y qué parte a lo que Q-learning no ha aprendido?

2. **$\varepsilon$ decreciente frente a $\varepsilon$ constante.** Entrena con
   $\varepsilon = 0{,}1$ fijo y compara la curva con la del esquema decreciente. ¿Cuál
   aprende antes? ¿Cuál termina mejor? ¿Por qué?

3. **El papel de $\gamma$.** Prueba $\gamma \in \{0{,}5;\ 0{,}9;\ 0{,}95;\ 0{,}99\}$ en una
   instancia M. ¿Qué le pasa a la política con $\gamma$ pequeño? Relaciónalo con la distancia
   entre el origen y el destino.

4. **El papel de $\alpha$.** Con $\alpha$ grande el agente aprende rápido y olvida rápido.
   Compruébalo: da una curva con $\alpha = 0{,}8$ y otra con $\alpha = 0{,}05$.

5. **Estados visitados.** Compara el número de estados que aparecen en tu tabla Q con el
   número de posiciones transitables de la instancia. ¿Por qué no coinciden? ¿Es un problema?
   Usa el mapa de la política para ver cuáles faltan y dónde están.

6. **Reward hacking.** Premiando el paso, el agente deja de ir al destino en cuanto la
   recompensa por paso es lo bastante alta. Describe qué hace exactamente y por qué es
   perfectamente racional dadas las recompensas. Luego afina: dar vueltas para siempre
   cobrando $r$ por paso vale $r/(1-\gamma)$, y llegar vale +100. Con $\gamma = 0{,}95$,
   ¿a partir de qué $r$ compensa no llegar nunca? Comprueba tu predicción con el entorno, y
   explica por qué con $r = +1$ basta con subir $\gamma$ a 0,99 para que el agente se quede
   dando vueltas. (En el entorno nada dura para siempre: el episodio se corta al agotar
   `limite_pasos`. ¿Cambia eso la conclusión?) ¿Qué dice esto sobre especificar objetivos
   para un sistema de IA?

7. **Política frente a tabla Q.** Dos entrenamientos con semillas distintas dan tablas Q
   distintas. ¿Pueden dar la misma política? ¿Y políticas distintas con la misma calidad?
   Compruébalo con tres semillas, `evaluar_politica` y `dibujar_politica(..., comparar_con=...)`.

8. **Cómo se propaga el valor.** En el mapa de valor del apartado 9.2, ¿por qué con pocos
   episodios el valor alto aparece solo cerca del destino? ¿Cómo llega a los otros niveles?
   Relaciónalo con la regla de actualización de Q-learning y con $\gamma$, y explica qué
   consecuencia tiene para la política en las posiciones más alejadas del destino.

9. **Lo aprendido frente a lo óptimo.** ¿Qué fracción de la recompensa media de la política
   óptima consigue tu Q-learning en cada instancia? En el mapa con `comparar_con`, ¿dónde
   están las posiciones en que las dos políticas difieren: en el recorrido habitual o lejos
   de él? ¿Por qué tantas diferencias cambian tan poco la recompensa?

10. **Óptima con resbalones frente a A$^*$.** ¿Encuentras en el mapa de iteración de valores
    alguna posición donde la política óptima no siga el camino más corto de la PR1? Explica
    esa decisión con las transiciones del modelo (qué pasa si resbala junto a un mamparo o
    a una zona cara). Si no encuentras ninguna, explica por qué en esta estación el riesgo
    de resbalar apenas cambia el camino.

## 10. Ampliaciones

**Evaluación no continua: esta sección es obligatoria.** Se dispone de más tiempo para
resolver y entregar la práctica, así que la entrega incluye además lo que se pide aquí.
**Evaluación continua: es voluntaria**, y lo que se entregue se valora en el apartado de
código.

Para cada una, exporta los ficheros de solución con el nombre de algoritmo que se indica y
añade al cuaderno una tabla y un párrafo con lo que observas.

* **SARSA.** Implementa SARSA y compárala con Q-learning en las tres instancias: curva,
  tasa de éxito y coste medio. SARSA aprende la política que realmente sigue, incluida la
  exploración; Q-learning aprende la voraz. ¿Se nota la diferencia en un entorno donde
  resbalas? Algoritmo: `sarsa`.
* **Inicialización optimista.** En lugar de empezar con Q a cero, empieza con un valor alto.
  Explica por qué eso explora sin necesidad de $\varepsilon$, y mide cuánto tarda en
  converger frente al $\varepsilon$-voraz. Algoritmo: `q_learning_optimista`.

## 11. Referencias

* S. Russell y P. Norvig. *Artificial Intelligence: A Modern Approach*, 4.ª ed., Pearson,
  2021. Capítulo 22.
* R. S. Sutton y A. G. Barto. *Reinforcement Learning: An Introduction*, 2.ª ed., MIT Press,
  2018. Capítulos 5 y 6.
* Transparencias del tema 5 de la asignatura.
